In [1]:
import sys, torch, faultdet
print(sys.executable)
print('torch', torch.__version__, '| CUDA', torch.version.cuda)
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print('V100 (sm_70) supported:', 'sm_70' in torch.cuda.get_arch_list())
a = torch.randn(2000, 2000, device='cuda'); print('GPU compute OK:', torch.isfinite(a @ a).all().item())

c:\ProgramData\anaconda3\envs\faultdet\python.exe
torch 2.14.1+cu126 | CUDA 12.6
GPUs: ['Tesla V100-PCIE-32GB', 'Tesla V100-PCIE-32GB']
V100 (sm_70) supported: True
GPU compute OK: True


In [2]:
%run scripts/prepare_data.py paper

data\raw\paper -> data\processed\paper
  197/197 runs
paper     train 2355  test 785


In [3]:
%run scripts/prepare_data.py varied

data\raw\varied -> data\processed\varied
  500/4000 runs
  1000/4000 runs
  1500/4000 runs
  2000/4000 runs
  2500/4000 runs
  3000/4000 runs
  3500/4000 runs
  4000/4000 runs
random    train 2890  val 510  test 600
ood_rf    train 2346  val 415  test 813
ood_load  train 2260  val 399  test 1007


In [4]:
from faultdet.data import load_split, feature_scales, FaultWindows
root = 'data/processed/paper'
sp = load_split(root, 'paper')
sc = feature_scales(root, sp['train'][0], 'I')
ds = FaultWindows(root, *sp['train'], 'I', sc, ('uniform', 5.0))
x, yt, yl = ds[0]
print(len(ds), tuple(x.shape), yt, yl, sc, float(x.abs().max()))

2355 (12800, 15) 0 0 {'I': 1909.3990478515625} 0.09520938992500305


In [61]:
import glob, os
for f in sorted(glob.glob('logs/*.txt'), key=os.path.getmtime)[-2:]:
    lines = open(f).read().splitlines()
    ep = [l for l in lines if l.startswith('ep ')]
    print(f'{os.path.basename(f)}  |  {len(ep)} epochs done')
    for l in ep[-3:]:
        print('    ', l)

dataset-varied_split-ood_rf_feat-dI_seed-0.txt  |  112 epochs done
     ep 110  train 0.2849 (0.943/0.970)  val 0.4410 (0.923/0.933)  14s
     ep 111  train 0.2169 (0.945/0.974)  val 0.2514 (0.916/0.978)  14s
     ep 112  train 0.2224 (0.942/0.975)  val 0.2572 (0.920/0.961)  14s
dataset-varied_split-ood_rf_feat-I_dI_seed-0.txt  |  53 epochs done
     ep  51  train 0.4620 (0.913/0.901)  val 0.3378 (0.928/0.923)  19s
     ep  52  train 0.5199 (0.909/0.887)  val 0.7431 (0.906/0.829)  19s
     ep  53  train 0.4483 (0.909/0.910)  val 0.3326 (0.916/0.928)  20s


In [27]:
import glob, json
for f in sorted(glob.glob('results/varied_*.json')):
    r = json.load(open(f)); h = r['history']
    print(f"{r['tag']:48s} epochs {len(h):3d}  best ep {r['best_epoch']:3d}  "
          f"max val loc {max(e['val_loc_acc'] for e in h):.3f}  |  "
          f"test type(m) {r['type_merged']['acc']:.3f}  loc {r['loc']['acc']:.3f}")

varied_ood_load_Xception_I_dI_V_dV_f1_s0         epochs 118  best ep 103  max val loc 0.880  |  test type(m) 0.955  loc 0.837
varied_ood_load_Xception_I_dI_dV_f1_s0           epochs  24  best ep   9  max val loc 0.063  |  test type(m) 0.938  loc 0.063
varied_ood_load_Xception_I_dI_f1_s0              epochs 150  best ep 143  max val loc 0.850  |  test type(m) 0.945  loc 0.843
varied_ood_load_Xception_I_f1_s0                 epochs  79  best ep  64  max val loc 0.847  |  test type(m) 0.956  loc 0.848
varied_ood_load_Xception_dI_f1_s0                epochs 117  best ep 102  max val loc 0.960  |  test type(m) 0.935  loc 0.943
varied_ood_rf_Xception_I_dI_V_dV_f1_s0           epochs  89  best ep  74  max val loc 0.875  |  test type(m) 0.659  loc 0.776
varied_ood_rf_Xception_I_dI_dV_f1_s0             epochs  31  best ep  16  max val loc 0.123  |  test type(m) 0.798  loc 0.137
varied_ood_rf_Xception_I_dI_f1_s0                epochs  80  best ep  65  max val loc 0.935  |  test type(m) 0.720  lo

In [28]:
import glob, json
for f in sorted(glob.glob('results/varied_*.json')):
    r = json.load(open(f)); h = r['history']
    print(f"{r['tag']:42s} ep {len(h):3d} best {r['best_epoch']:3d}  "
          f"max val loc {max(e['val_loc_acc'] for e in h):.3f}  test loc {r['loc']['acc']:.3f}  "
          f"scales {({k: round(v, 1) for k, v in r['scales'].items()})}")

varied_ood_load_Xception_I_dI_V_dV_f1_s0   ep 118 best 103  max val loc 0.880  test loc 0.837  scales {'I': 3480.8, 'dI': 3408.4, 'V': 201.8, 'dV': 219.3}
varied_ood_load_Xception_I_dI_dV_f1_s0     ep  24 best   9  max val loc 0.063  test loc 0.063  scales {'I': 3480.8, 'dI': 3408.4, 'dV': 219.3}
varied_ood_load_Xception_I_dI_f1_s0        ep 150 best 143  max val loc 0.850  test loc 0.843  scales {'I': 3480.8, 'dI': 3408.4}
varied_ood_load_Xception_I_f1_s0           ep  79 best  64  max val loc 0.847  test loc 0.848  scales {'I': 3480.8}
varied_ood_load_Xception_dI_f1_s0          ep 117 best 102  max val loc 0.960  test loc 0.943  scales {'dI': 3408.4}
varied_ood_rf_Xception_I_dI_V_dV_f1_s0     ep  89 best  74  max val loc 0.875  test loc 0.776  scales {'I': 3480.8, 'dI': 3408.4, 'V': 199.2, 'dV': 219.3}
varied_ood_rf_Xception_I_dI_dV_f1_s0       ep  31 best  16  max val loc 0.123  test loc 0.137  scales {'I': 3480.8, 'dI': 3408.4, 'dV': 219.3}
varied_ood_rf_Xception_I_dI_f1_s0        

In [2]:
%run scripts/summarize.py

configuration                                                  n  type_merged_acc         type_acc         type_mcc          loc_acc          loc_mcc      params
varied_ood_load_DenseNet169_I_f1                               2      0.954±0.009      0.884±0.009      0.876±0.010      0.946±0.012      0.942±0.013     15.587M
varied_ood_load_DenseNet169_dI_f1                              2      0.944±0.008      0.874±0.007      0.867±0.005      0.962±0.016      0.959±0.017     15.587M
varied_ood_load_Lite-c1-sn-ls0-d100-w0-nzlow_I_dI_f1           3      0.931±0.007      0.866±0.007      0.857±0.008      0.940±0.009      0.936±0.010      0.168M
varied_ood_load_Xception_I_dI_V_dV_f1                          3      0.940±0.007      0.865±0.008      0.856±0.009      0.900±0.030      0.894±0.032     19.979M
varied_ood_load_Xception_I_dI_f1                               3      0.946±0.008      0.877±0.012      0.868±0.013      0.946±0.018      0.943±0.019     19.977M
varied_ood_load_Xception_I_f

In [54]:
import glob, json
for f in sorted(glob.glob('results/varied_*.json')):
    r = json.load(open(f)); h = r['history']
    print(f"{r['tag']:46s} test loc {r['loc']['acc']:.3f}  max train loc {max(e['train_loc_acc'] for e in h):.3f}  "
          f"max val loc {max(e['val_loc_acc'] for e in h):.3f}  best ep {r['best_epoch']}")

varied_ood_load_Xception_I_dI_V_dV_f1_s0       test loc 0.819  max train loc 0.754  max val loc 0.820  best ep 138
varied_ood_load_Xception_I_dI_V_dV_f1_s1       test loc 0.063  max train loc 0.087  max val loc 0.063  best ep 55
varied_ood_load_Xception_I_dI_V_dV_f1_s2       test loc 0.904  max train loc 0.966  max val loc 0.930  best ep 88
varied_ood_load_Xception_I_dI_dV_f1_s0         test loc 0.088  max train loc 0.095  max val loc 0.083  best ep 116
varied_ood_load_Xception_I_dI_dV_f1_s1         test loc 0.195  max train loc 0.186  max val loc 0.178  best ep 147
varied_ood_load_Xception_I_dI_dV_f1_s2         test loc 0.893  max train loc 0.942  max val loc 0.942  best ep 109
varied_ood_load_Xception_I_dI_f1_s0            test loc 0.925  max train loc 0.969  max val loc 0.945  best ep 147
varied_ood_load_Xception_I_dI_f1_s1            test loc 0.924  max train loc 0.980  max val loc 0.975  best ep 95
varied_ood_load_Xception_I_dI_f1_s2            test loc 0.941  max train loc 0.969 

In [60]:
%run scripts/summarize.py results_fix

configuration                                           n  type_merged_acc         type_acc         type_mcc          loc_acc          loc_mcc
varied_random_Xception_I_dI_V_dV_f1                     4      0.948±0.006      0.877±0.010      0.869±0.011      0.942±0.012      0.938±0.013
varied_random_Xception_I_f1                             2      0.953±0.001      0.880±0.002      0.874±0.006      0.961±0.011      0.958±0.011
varied_random_Xception_dI_f1                            4      0.948±0.002      0.879±0.004      0.874±0.002      0.980±0.008      0.979±0.009


In [68]:
import glob, json, numpy as np
for feat in ('I', 'dI'):
    cm = sum(np.array(json.load(open(f))['cm_type']) for f in glob.glob(f'results/varied_ood_rf_Xception_{feat}_f1_s*.json'))
    print(feat, 'LG -> HIF errors:', cm[1:4, 12:15].sum(), '/', cm[1:4].sum(), '   LLG -> LL:', cm[7:10, 4:7].sum(), '/', cm[7:10].sum())

I LG -> HIF errors: 398 / 429    LLG -> LL: 126 / 468
dI LG -> HIF errors: 401 / 429    LLG -> LL: 286 / 468


In [3]:
%run scripts/summarize.py

configuration                                                  n  type_merged_acc         type_acc         type_mcc          loc_acc          loc_mcc      params
varied_ood_load_DenseNet169_I_f1                               2      0.954±0.009      0.884±0.009      0.876±0.010      0.946±0.012      0.942±0.013     15.587M
varied_ood_load_DenseNet169_dI_f1                              2      0.944±0.008      0.874±0.007      0.867±0.005      0.962±0.016      0.959±0.017     15.587M
varied_ood_load_Lite-c0-gn-ls0-d100-w0-nzlow_I_f1              3      0.955±0.005      0.886±0.006      0.879±0.007      0.920±0.004      0.914±0.004      0.165M
varied_ood_load_Lite-c0-sn-ls0-d100-w0-nzlow_I_dI_f1           3      0.938±0.008      0.873±0.008      0.866±0.009      0.924±0.018      0.919±0.020      0.168M
varied_ood_load_Lite-c1-gn-ls0-d100-w0-nzlow_I_dI_f1           3      0.953±0.005      0.880±0.011      0.871±0.012      0.948±0.018      0.944±0.020      0.168M
varied_ood_load_Lite-c1-sn-l

In [4]:
%run scripts/errors.py

configuration                                                  n    LG>HIF   HIF>LG   LLG>LL   LL>LLG    LG ph    LL ph   LLG ph   HIF ph  NF miss false NF
varied_ood_load_DenseNet169_I_f1                               2     17.1%     2.2%     1.5%     0.0%     0.0%     0.0%     0.0%     0.2%     0.0%     0.0%
varied_ood_load_DenseNet169_dI_f1                              2     18.7%     4.3%     1.0%     0.0%     0.0%     0.0%     0.0%     0.5%     0.0%     0.0%
varied_ood_load_Lite-c0-gn-ls0-d100-w0-nzlow_I_f1              3     16.7%     2.1%     0.5%     0.8%     0.0%     0.0%     0.0%     0.0%     0.0%     0.0%
varied_ood_load_Lite-c0-sn-ls0-d100-w0-nzlow_I_dI_f1           3     24.3%     2.2%     0.3%     0.3%     0.0%     0.0%     0.0%     0.0%     0.0%     0.0%
varied_ood_load_Lite-c1-gn-ls0-d100-w0-nzlow_I_dI_f1           3     17.2%     1.3%     1.7%     0.3%     0.0%     0.0%     0.0%     0.0%     0.0%     0.0%
varied_ood_load_Lite-c1-sn-ls0-d100-w0-nzfull_I_dI_f1          3

In [5]:
%run scripts/classical.py

=== varied_random_GBM-seqharm-d100_I_dI_f1_s0 (157 features, 0.4 min) ===
type  acc 0.9033  F1 0.9032  MCC 0.8963  AUC 0.9929
loc   acc 1.0000  F1 1.0000  MCC 1.0000  AUC 1.0000
type, LLL/LLLG merged: acc 0.9617  MCC 0.9586
=== varied_ood_load_GBM-seqharm-d100_I_dI_f1_s0 (157 features, 0.3 min) ===
type  acc 0.8788  F1 0.8804  MCC 0.8703  AUC 0.9858
loc   acc 0.9950  F1 0.9950  MCC 0.9947  AUC 0.9999
type, LLL/LLLG merged: acc 0.9474  MCC 0.9432
=== varied_ood_rf_GBM-seqharm-d100_I_dI_f1_s0 (157 features, 0.3 min) ===
type  acc 0.7466  F1 0.7037  MCC 0.7311  AUC 0.9322
loc   acc 0.9520  F1 0.9509  MCC 0.9491  AUC 0.9995
type, LLL/LLLG merged: acc 0.8192  MCC 0.8089
=== varied_random_GBM-seqharm-d100_I_dI_f1_s1 (157 features, 0.4 min) ===
type  acc 0.9033  F1 0.9032  MCC 0.8964  AUC 0.9924
loc   acc 0.9983  F1 0.9983  MCC 0.9982  AUC 1.0000
type, LLL/LLLG merged: acc 0.9667  MCC 0.9640
=== varied_ood_load_GBM-seqharm-d100_I_dI_f1_s1 (157 features, 0.3 min) ===
type  acc 0.8868  F1 0.887

In [ ]:
%run scripts/latency.py --gpu 0

model                              params       MACs    CPU x1 ms   GPU ms
Xception (paper)                  19.975M   13.5670G       344.89    26.90
DenseNet-169 (paper)              15.587M    7.8993G       515.04    64.98
Inception-ResNet-v2 (paper)       48.088M   23.1678G      1319.70    83.56
EfficientNet-B2 (paper)           10.833M    2.9370G       355.09    33.87
Lite (I+dI, 1 stream)              0.168M    0.0027G         4.03     3.53
Lite2 (two-stream)                 0.263M    0.0049G         9.13     7.06
Lite2, half cycle                  0.263M    0.0033G         8.86     7.08
saved results\latency.csv


: 

In [1]:
%run scripts/summarize.py

configuration                                                      n  type_merged_acc         type_acc         type_mcc          loc_acc          loc_mcc      params
paper_paper_DenseNet169_I_f1                                       1      1.000            0.934            0.929            0.995            0.995           15.587M
paper_paper_EfficientNetB2_I_f1                                    1      1.000            0.931            0.931            0.996            0.996           10.833M
paper_paper_InceptionResNetV2_I_f1                                 1      1.000            0.935            0.935            1.000            1.000           48.088M
paper_paper_Lite-c1-sn-ls0-d100-w0-nzlow_I_dI_f1                   3      1.000±0.000      0.934±0.002      0.932±0.005      1.000±0.000      1.000±0.000      0.168M
paper_paper_Lite2-tI+dI-ldI-c1-sn-ls0-d100-w0-nzlow_I_dI_f1        3      1.000±0.000      0.935±0.001      0.934±0.001      1.000±0.000      1.000±0.000      0.263M
pape

In [2]:
%run scripts/summarize.py --val

configuration                                                      n     val_type_acc      val_loc_acc         val_loss      params
paper_paper_DenseNet169_I_f1                                       1      0.934            0.995            0.115           15.587M
paper_paper_EfficientNetB2_I_f1                                    1      0.931            0.996            0.106           10.833M
paper_paper_InceptionResNetV2_I_f1                                 1      0.935            1.000            0.098           48.088M
paper_paper_Lite-c1-sn-ls0-d100-w0-nzlow_I_dI_f1                   3      0.934±0.002      1.000±0.000      0.093±0.000      0.168M
paper_paper_Lite2-tI+dI-ldI-c1-sn-ls0-d100-w0-nzlow_I_dI_f1        3      0.935±0.001      1.000±0.000      0.093±0.000      0.263M
paper_paper_Xception_I_f1                                          1      0.931            1.000            0.095           19.975M
varied_ood_load_DenseNet169_I_f1                                   1      0.

In [3]:
%run scripts/errors.py --match Lite2

configuration                                                      n    LG>HIF   HIF>LG   LLG>LL   LL>LLG    LG ph    LL ph   LLG ph   HIF ph  NF miss false NF
paper_paper_Lite2-tI+dI-ldI-c1-sn-ls0-d100-w0-nzlow_I_dI_f1        3      0.0%     0.0%     0.0%     0.0%     0.0%     0.0%     0.0%     0.0%     0.0%     0.0%
varied_ood_load_Lite2-tI+dI-ldI-c1-sn-ls0-d100-w0-nzlow-shufloc_I_dI_f1 3     19.3%     1.1%     0.2%     0.0%     0.0%     0.0%     0.0%     0.0%     0.0%     0.1%
varied_ood_load_Lite2-tI+dI-ldI-c1-sn-ls0-d100-w0-nzlow_I_dI_f0.25 3     27.9%     3.5%     0.8%     0.6%     0.0%     0.0%     0.0%     0.0%     0.0%     0.0%
varied_ood_load_Lite2-tI+dI-ldI-c1-sn-ls0-d100-w0-nzlow_I_dI_f0.5  3     17.8%     6.7%     0.2%     0.0%     0.0%     0.0%     0.0%     0.0%     0.0%     0.1%
varied_ood_load_Lite2-tI+dI-ldI-c1-sn-ls0-d100-w0-nzlow_I_dI_f1    3     19.9%     3.0%     0.0%     0.0%     0.0%     0.0%     0.0%     0.0%     0.0%     0.0%
varied_ood_load_Lite2-tI+dI-ldI-c1-

In [6]:
%run scripts/paper_tables.py

### Reproduction under the original (leaky) protocol

| | Params, M: published | Params, M: ours | Type & phase acc., %: published | Type & phase acc., %: ours | Type & phase acc., %: ours, merged | Location acc., %: published | Location acc., %: ours |
|---|---|---|---|---|---|---|---|
| Xception | 21.5 | 20.0 | 98.6 | 93.1 | **100.0** | 98.1 | **100.0** |
| DenseNet-169 | 17.1 | 15.6 | 98.5 | 93.4 | **100.0** | 99.4 | 99.5 |
| Inception-ResNet-v2 | 49.6 | 48.1 | 98.2 | **93.5** | **100.0** | 97.6 | **100.0** |
| EfficientNet-B2 | 12.4 | 10.8 | 97.7 | 93.1 | **100.0** | 97.1 | 99.6 |
| Lite (ours) | – | 0.17 | – | 93.4 ± 0.2 | **100.0 ± 0.0** | – | **100.0 ± 0.0** |
| Lite2 (ours) | – | 0.26 | – | **93.5 ± 0.1** | **100.0 ± 0.0** | – | **100.0 ± 0.0** |

Original protocol: 15 noisy copies per simulation (200 for no-fault), copy-level 75/25 split, validation = test. Merged: ABC and ABCG counted as one class (indistinguishable from phase currents). Our ports of the paper's models have 1

In [2]:
%run scripts/paper_tables.py

Exception: File `'scripts/paper_tables.py'` not found.